# Modeling Water Fluxes In The Soil-Plant System

## Soil hydraulics with Dumux-rosi

In [ ]:
# generic Python libraries
import os;
import sys;
import numpy as np;
import matplotlib.pyplot as plt;
from IPython.display import SVG, Image, display # to show svg files in the notebook

# specific CPlantBox libraries
search_path = '../lib';
sys.path.append(search_path);
import plantbox as pb; #CPlantBox itself
import visualisation.vtk_plot as vp # for quick vizualisations 

In [ ]:
from rosi_richards import RichardsSP # C++ part 
from dumux_rosi.richards_no_mpi import RichardsNoMPIWrapper as RichardsWrapper # Python part  
import dumux_rosi.van_genuchten as vg  

### Size and Resolution of the domain, initial condition

**Q: can you explaiun why the soil below is in 1D?**\
**Q: increase the soil resolution, so that we have 1 cell for 1cm of depth.**\
**Q: keep the same resolution, but change the soil dimention, so that is has a surface of 10 x 10 cm2 and a depth of 100cm.**\
**Q: Change the initial mean soil water potential from -400cm to -1000cm.**

In [ ]:

min_b = [-50., -5., -500.] # size [cm]  
max_b = [50., 5., 0.] # size [cm]  
cell_number = [1, 1, 5] # resolution <= 1D
loam = [0.08, 0.43, 0.04, 1.6, 50]
s = RichardsWrapper(RichardsSP()) 
s.initialize()     
s.createGrid( min_b, max_b, cell_number)      
s.setHomogeneousIC(-400., equilibrium = True)  # cm pressure head   
s.setVGParameters([loam])
s.setTopBC("noFlux")  
s.setBotBC("noFlux")
s.initializeProblem()      
vp.plot_soil(s, "pressure head", min_b, max_b, cell_number, interactiveImage = False)


### The Van Genuchten Model

The constitutive equations are described the Van Genuchten model, giving the soil retention curve, hydraulic conductivities or matrix flux potentials. The following script show example parametrisations of various soil types. \
**Q: looking at the soil below, can you explain which will oppose the most resistance to water loss?**


In [ ]:
# Define van Genuchten parameters for sand, loam and clay  
# theta_r (-), theta_s (-), alpha (1/cm), n (-), Ks (cm d-1)
sand =  vg.Parameters([0.045, 0.43, 0.15, 3, 1000])
loam =  vg.Parameters([0.08, 0.43, 0.04, 1.6, 50])
clay =  vg.Parameters([0.1, 0.4, 0.01, 1.1, 10])

print("theta_R", sand.theta_R, "theta_S",sand.theta_S, "alpha", sand.alpha,"n",sand.n, 'm',sand.m,'Ksat', sand.Ksat)

vg.plot_retention_curve(sand, label_ = "sand")
vg.plot_retention_curve(loam, label_ = "loam")
vg.plot_retention_curve(clay, label_ = "clay")
plt.legend()
plt.show()

Therefore, the retention curve relates matric potential and water content and vice versa: 

In [ ]:
theta = 0.3 # in cm3 water /cm3 soil
print(vg.pressure_head(theta, sand))
p_head = vg.pressure_head(theta, sand) # in cm
print(vg.water_content(p_head, sand)) # same as the initial theta    

**Q: Compute the plant avaiable water (in cm3 water /cm3 soil) of the loamy soil above.**

In [ ]:
wilting_point = -15000 # cm
field_capacity = -100 # cm
####
#
# PAW = 
#
####

### Boundary conditions
1. Neumann boundary conditions "noFlux", "constantFlux" (flux [cm/day], volume per surface per day)
2. Dirichlet boundary conditions "constantPressure" (matric potential [cm])

ATT: some values _could_ lead to non-convergence or hanging. If the solver hangs, restart the kernel.

**Q: Set a flux of -1cm/day at the top boundary and a Dirichlet boundary confition of 0. cm at the bottom**\
Hint: the commands are already written. You need to select which lines to uncomment.\
**Q: A positive Neumann boundary condition represent a gain or a loss of water for the soil?**\
**Q: With the intial soil defined below, an upper Dirichlet boundary condition of -1500 and a lower "noFlux" Neumann boundary condition lead to a loss or a gain of water for the soil?**\
**Q: using the Dirichlet boundary conditions at teh top and bottom boundary, which value can you set to obtain a higher pressure at the bottom compaired with the top.**

In [ ]:
top_value = 0.; bot_value = 0. # default

# Neumann BC. flux is lowered if it cannot be respected
top_boundary_condition = "noFlux"   
# top_boundary_condition = "constantFlux"; top_value = -1. # ; top_value = 1. #[cm/day]

bot_boundary_condition = "noFlux"   
# bot_boundary_condition = "constantFlux"; bot_value = 1. # [cm/day]

# Dirichlet BC.
# top_boundary_condition = "constantPressure"; top_value = -1500. # [cm]
# bot_boundary_condition = "constantPressure"; bot_value = 0. # [cm]

initial_pressure = -400
at_equilibrium = True

min_b = [-1., -1., -150.] # size [cm]  
max_b = [0., 0., 0.] # size [cm]  
cell_number = [1, 1, 150] # resolution
loam = [0.08, 0.43, 0.04, 1.6, 50]
s = RichardsWrapper(RichardsSP()) 
s.initialize()     
s.createGrid( min_b, max_b, cell_number)      
s.setHomogeneousIC(initial_pressure, equilibrium = at_equilibrium)  # cm pressure head   
s.setVGParameters([loam])

###
s.setTopBC(top_boundary_condition, top_value)  
s.setBotBC(bot_boundary_condition, bot_value)
###

s.initializeProblem()   
points = s.getDofCoordinates()
theta0 = s.getWaterContent()
pressure0 = s.getSolutionHead()
s.solve(10.) # in days
theta = s.getWaterContent() 
pressure = s.getSolutionHead()

plt.figure(0)
plt.plot(theta0, points[:, 2], linewidth = 2, label = "initial")
plt.plot(theta, points[:, 2], linewidth = 2, label = "final")
plt.xlabel(r'$\theta$ (cm$^3$ cm$^{-3}$)', fontsize = 18)
plt.ylabel('depth (cm)', fontsize = 18)
plt.xticks(fontsize = 14); plt.yticks(fontsize = 14)
plt.xlim(loam[0], loam[1]) 
plt.legend()
plt.show() 

plt.figure(0)
plt.plot(pressure0, points[:, 2], linewidth = 2, label = "initial")
plt.plot(pressure, points[:, 2], linewidth = 2, label = "final")
plt.xlabel(r'pressure head (cm)', fontsize = 18)
plt.ylabel('depth (cm)', fontsize = 18)
plt.xticks(fontsize = 14); plt.yticks(fontsize = 14)
plt.legend()
plt.show() 

### Simulation loop
Complete the code below, to :
- simulate the soil water flux for 50 days. Hint: replace one of the ___ signs.
- set a time step of 1 day. Hint: replace one of the ___ signs.
- set a constant flux of -10 cm/day on the top boundary condition for the first 49 days of simulation and then set the top boundary condition to +1 cm/day for the last day. Hint: Uncomment the lines in the simulation loop and replace the three ___ signs.

In [ ]:
top_boundary_condition = "constantFlux"; top_value = -1. #[cm/day]
bot_boundary_condition = "noFlux"   

initial_pressure = -400
at_equilibrium = True

min_b = [-1., -1., -150.] # size [cm]  
max_b = [0., 0., 0.] # size [cm]  
cell_number = [1, 1, 150] # resolution
loam = [0.08, 0.43, 0.04, 1.6, 50]
s = RichardsWrapper(RichardsSP()) 
s.initialize()     
s.createGrid( min_b, max_b, cell_number)      
s.setHomogeneousIC(initial_pressure, equilibrium = at_equilibrium)  # cm pressure head   
s.setVGParameters([loam])


s.initializeProblem()   
dt = ___ # day
simtime = ___ # day
N = int(simtime/dt) # number of time steps


points = s.getDofCoordinates()
theta0 = s.getWaterContent()
pressure0 = s.getSolutionHead()

current_simtime = 0.
for i in range(N):
    print('day', current_simtime, end=', ')    
    #if current_simtime < ___ :
    #    s.setTopBC(top_boundary_condition, ___) 
    #else:
    #    s.setTopBC(top_boundary_condition, ___) 
        
    s.solve(dt) # in days
    current_simtime += dt
    
theta = s.getWaterContent() 
pressure = s.getSolutionHead()

plt.figure(0)
plt.plot(theta0, points[:, 2], linewidth = 2, label = "initial")
plt.plot(theta, points[:, 2], linewidth = 2, label = "final")
plt.xlabel(r'$\theta$ (cm$^3$ cm$^{-3}$)', fontsize = 18)
plt.ylabel('depth (cm)', fontsize = 18)
plt.xticks(fontsize = 14); plt.yticks(fontsize = 14)
plt.xlim(loam[0], loam[1]) 
plt.legend()
plt.show() 

plt.figure(0)
plt.plot(pressure0, points[:, 2], linewidth = 2, label = "initial")
plt.plot(pressure, points[:, 2], linewidth = 2, label = "final")
plt.xlabel(r'pressure head (cm)', fontsize = 18)
plt.ylabel('depth (cm)', fontsize = 18)
plt.xticks(fontsize = 14); plt.yticks(fontsize = 14)
plt.legend()
plt.show() 